# RTMA hourly EMC export

Pulls NWS Real-Time Mesoscale Analysis (`NOAA/NWS/RTMA`, hourly 2.5 km, 2011–present),
computes **per-pixel** RH, VPD, and EMC (so the nonlinear EMC(T, RH) is not biased by
`EMC(mean(T), mean(RH))`), then reduces to pyrome-mean hourly scalars and exports CSVs.

The downstream `fb_tools.weather.rtma` module consumes these CSVs to produce
FlamMap percentile-scenario FM (FM1/FM10/FM100 via hourly NFDRS78 time-lag with
Bradshaw 1984 precip saturation-stall; FM_herb/FM_woody via GSI).

**Schema** (per export task — one per year, all 9 CO pyromes):

```
pyrome_id, datetime_utc, tmp_f, rh_pct, emc_pct, vpd_pa, pcp_mm_hr
```

ERC stays GridMET-derived (FSPro/FSim daily contract). HRRR stays the wind source.
This pipeline only upgrades the dead-FM and (optionally) live-FM legs of the
per-pyrome FlamMap scenarios.

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='cfri-ee')
print('GEE authenticated.')

## Pyrome geometries (CO analysis extent)

The CO pyromes per `CLAUDE.md`: `42, 43, 45, 46, 47, 52, 53, 56, 128`.

In [ ]:
CO_PYROME_IDS = [42, 43, 45, 46, 47, 52, 53, 56, 128]

pyromes = ee.FeatureCollection('projects/cfri-ee/assets/weather/Pyromes_CONUS_20200206')
co_pyromes = pyromes.filter(ee.Filter.inList('PYROME', CO_PYROME_IDS))
print('CO pyromes:', co_pyromes.size().getInfo())
co_pyromes.aggregate_array('PYROME').getInfo()

## RTMA ImageCollection — inspect bands

RTMA on GEE exposes `TMP` (2-m temp, **°C**), `DPT` (dew point, **°C**), `SPFH`
(specific humidity, kg/kg), and `ACPC01` (hourly accumulated precip, kg/m² ≡ mm).
Note: the GEE catalog lists TMP and DPT in °C — **not Kelvin** — despite some
third-party examples treating them as K. Confirm with the range check below.

In [ ]:
rtma = ee.ImageCollection('NOAA/NWS/RTMA')
sample_img = rtma.filterDate('2020-07-15', '2020-07-16').first()
print('Bands:', sample_img.bandNames().getInfo())
print('Sample date:', ee.Date(sample_img.get('system:time_start')).format('YYYY-MM-dd HH:mm').getInfo())

# Sanity-check TMP range over CONUS — expect ~-40 to +45 °C, NOT 230–320 K
tmp_stats = sample_img.select('TMP').reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=sample_img.geometry(),
    scale=10000,
    bestEffort=True,
).getInfo()
print('TMP min/max (should be °C, not K):', tmp_stats)

## Per-pixel transforms — RH, VPD, EMC

All computed on RTMA's native grid *before* the pyrome-mean reduction so the
nonlinear NFDRS EMC function isn't biased.

- RH from T, T_d via Magnus saturation vapor pressure ratio
- VPD = e_s(T) − e_s(T_d), in Pa
- EMC: three-regime piecewise NFDRS function (Cohen & Deeming 1985), see
  `fb_tools/weather/nfdrs.py:calc_emc` for the local equivalent.

In [ ]:
TMP_BAND = 'TMP'      # 2-m temperature, °C (NOT Kelvin — GEE RTMA spec)
DPT_BAND = 'DPT'      # 2-m dew point temperature, °C (NOT Kelvin)
PCP_BAND = 'ACPC01'   # 1-hour accumulated precipitation, kg/m² ≡ mm

def add_derived_bands(img):
    """Append tmp_f, rh_pct, vpd_pa, emc_pct, pcp_mm_hr per-pixel bands."""
    # TMP and DPT are already in °C per the RTMA band specification.
    t_c  = img.select(TMP_BAND)
    td_c = img.select(DPT_BAND)

    # Magnus saturation vapor pressure (hPa)
    es_t  = t_c.multiply(17.67).divide(t_c.add(243.5)).exp().multiply(6.112)
    es_td = td_c.multiply(17.67).divide(td_c.add(243.5)).exp().multiply(6.112)
    rh    = es_td.divide(es_t).multiply(100).clamp(0, 100).rename('rh_pct')
    # VPD in Pa (1 hPa = 100 Pa)
    vpd_pa = es_t.subtract(es_td).multiply(100).max(0).rename('vpd_pa')

    tmp_f = t_c.multiply(1.8).add(32).rename('tmp_f')

    # EMC three-regime (Cohen & Deeming 1985): R<10, 10<=R<50, R>=50
    emc_low  = rh.multiply(0.281073).subtract(tmp_f.multiply(rh).multiply(0.000578)).add(0.03229)
    emc_mid  = rh.multiply(0.160107).subtract(tmp_f.multiply(0.014784)).add(2.22749)
    emc_high = (
        rh.pow(2).multiply(0.005565)
          .subtract(tmp_f.multiply(rh).multiply(0.00035))
          .subtract(rh.multiply(0.483199))
          .add(21.0606)
    )
    emc = emc_high.where(rh.lt(50), emc_mid).where(rh.lt(10), emc_low).rename('emc_pct')

    # Server-side conditional — bandNames().contains() avoids a client-side .getInfo()
    # call that would fail inside map().
    pcp = ee.Image(ee.Algorithms.If(
        img.bandNames().contains(PCP_BAND),
        img.select(PCP_BAND).rename('pcp_mm_hr'),
        ee.Image.constant(0).rename('pcp_mm_hr'),
    ))

    return img.addBands([tmp_f, rh, vpd_pa, emc, pcp])

## Per-image pyrome-mean reduction

For each hourly image, reduceRegion over each pyrome geometry → one feature per
(pyrome, datetime_utc) with the scalar columns the local pipeline expects.

In [ ]:
REDUCE_BANDS = ['tmp_f', 'rh_pct', 'emc_pct', 'vpd_pa', 'pcp_mm_hr']
REDUCE_SCALE = 2500  # RTMA native resolution (m)

def reduce_to_pyromes(img):
    img = add_derived_bands(img).select(REDUCE_BANDS)

    def per_pyrome(feat):
        # Compute dt inside per_pyrome — keeps everything server-side
        dt = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss')
        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=feat.geometry(),
            scale=REDUCE_SCALE,
            maxPixels=1e9,
            bestEffort=True,
        )
        return ee.Feature(None, {
            'pyrome_id': feat.get('PYROME'),
            'datetime_utc': dt,
            'tmp_f':     stats.get('tmp_f'),
            'rh_pct':    stats.get('rh_pct'),
            'emc_pct':   stats.get('emc_pct'),
            'vpd_pa':    stats.get('vpd_pa'),
            'pcp_mm_hr': stats.get('pcp_mm_hr'),
        })

    return co_pyromes.map(per_pyrome)

## Quick test — single fire-season day

Sanity-check the reducer on one day before kicking off year-long exports.

In [ ]:
test_day = (
    rtma
    .filterDate('2020-07-15', '2020-07-16')
    .filterBounds(co_pyromes.geometry())
)
print('Hourly images in test day:', test_day.size().getInfo())

test_fc = ee.FeatureCollection(test_day.map(reduce_to_pyromes).flatten())
print('Features (24 hr × 9 pyromes ≈ 216):', test_fc.size().getInfo())

# Pull a small sample for inspection
test_fc.limit(24).getInfo()['features']

## Yearly fire-season exports (2011–2026)

One export task per year. Fire season = April 1 – October 31 (DOY 91–304, matches
GridMET conventions). Each task produces ~7 mo × 30 d × 24 h × 9 pyromes ≈ 45 k
rows. Submit all 16 tasks; monitor in the GEE Tasks tab.

Output: Google Drive folder `fb_tools_weather/`, prefix
`rtma_hourly_CO_pyromes_YYYY.csv`. Download locally and feed into
`fb_tools.weather.rtma.load_rtma_csv()`.

In [ ]:
EXPORT_FOLDER = 'fb_tools_weather'
EXPORT_PREFIX = 'rtma_hourly_CO_pyromes'
YEARS = range(2011, 2026) # 2011-2025 for complete yearly data

def submit_year(year):
    start = f'{year}-04-01' # 1April
    end = f'{year}-11-01'  # 1Nov
    season = (
        rtma.filterDate(start, end)
            .filterBounds(co_pyromes.geometry())
    )
    fc = ee.FeatureCollection(season.map(reduce_to_pyromes).flatten())

    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=f'{EXPORT_PREFIX}_{year}',
        folder=EXPORT_FOLDER,
        fileNamePrefix=f'{EXPORT_PREFIX}_{year}',
        fileFormat='CSV',
        selectors=['pyrome_id', 'datetime_utc', 'tmp_f', 'rh_pct', 'emc_pct', 'vpd_pa', 'pcp_mm_hr'],
    )
    task.start()
    return task

# Uncomment to submit. Run a single year first (e.g. 2020) and inspect the CSV
# before submitting the full 16-year batch.
tasks = [submit_year(y) for y in YEARS]
for t in tasks: print(t.status())

## Local-side: build per-pyrome FM and FlamMap scenarios

Once the CSVs are downloaded to a local directory (`./data/rtma_hourly/` below),
the local pipeline reads them with `load_rtma_csv()`, runs the NFDRS78 hourly lag
with FireFamilyPlus saturation-stall precip handling, collapses to daily peak-hour
(14:00 LST), and feeds the daily frame into the existing
`build_flammap_scenario_cache()` via `dead_fm_source='rtma'`.

In [ ]:
from pathlib import Path
import pandas as pd

from fb_tools.weather.rtma import (
    load_rtma_csv,
    build_rtma_dead_fm,
    build_rtma_live_fm,
    collapse_to_peak_hour,
)
from fb_tools.weather.gridmet import load_gridmet_csv, build_flammap_scenario_cache
from fb_tools.weather.hrrr import wind_percentiles_from_cell_cache

RTMA_DIR = Path('./data/rtma_hourly')          # directory of yearly CSVs
GRIDMET_CSV = Path('./data/gridmet_clim_CO_pyromes_fmask_pctiles.csv')
WIND_CACHE = Path('./data/hrrr_wind_cells/')
OUT_DIR = Path('./data/flammap_scenarios_rtma/')
CO_LAT = 39.5

rtma_hr = load_rtma_csv(RTMA_DIR)
print('RTMA hourly rows:', len(rtma_hr))
rtma_hr.head(3)

In [ ]:
# Hourly NFDRS78 lag with Bradshaw 1984 saturation-stall precip handling
rtma_dead = build_rtma_dead_fm(rtma_hr, precip_mode='stall', precip_threshold_mm_hr=0.25)

# Daily peak-hour collapse (14:00 LST = 21:00 UTC for MST)
rtma_peak = collapse_to_peak_hour(rtma_dead, peak_hour_local=14, tz_offset_hours=-7)

# GSI-based daily live FM (calibration caveat: Jolly thresholds; see CLAUDE.md)
rtma_live = build_rtma_live_fm(rtma_hr, lat_deg=CO_LAT, tz_offset_hours=-7)

# Join into a single daily frame indexed by (pyrome_id, date)
rtma_daily = rtma_peak.merge(
    rtma_live[['pyrome_id', 'date', 'gsi', 'FM_herb', 'FM_woody']],
    on=['pyrome_id', 'date'],
    how='left',
)
print('daily rows:', len(rtma_daily))
rtma_daily.head(3)

In [ ]:
# Feed into build_flammap_scenario_cache — ERC still GridMET-driven, HRRR winds,
# but dead and live FM swapped to RTMA-derived medians on the same ERC-band days.

gridmet = load_gridmet_csv(GRIDMET_CSV)
wind_pcts = wind_percentiles_from_cell_cache(WIND_CACHE)

scenarios = build_flammap_scenario_cache(
    gridmet,
    lat_deg=CO_LAT,
    wind_percentiles=wind_pcts,
    wind_direction=-2,
    dead_fm_source='rtma',
    live_fm_source='rtma',
    rtma_daily_df=rtma_daily,
    out_dir=OUT_DIR,
)
print('pyromes:', list(scenarios.keys()))
scenarios['46']['scenarios']